# FEF3001 · Assignment 1 · Is the rise of CO₂ accelerating?

**Part B — Colab.** This notebook uses the module `co2_analysis.py` that you built and tested in Antigravity (Part A).

1. Run the first code cell and upload **both** files from Part A, `co2_analysis.py` and `test_co2_analysis.py`, when asked.
2. Run every cell from top to bottom (*Runtime → Run all*). The second cell runs your tests; all of them must pass.
3. Write your answers in the **Your answer** cells. Fill in every `TODO`.
4. *Share → General access → Anyone with the link → Viewer*, then paste the link into the submission form.

**Data:** Lan, X., Tans, P. and K.W. Thoning, *Trends in globally-averaged CO₂ determined from NOAA Global Monitoring
Laboratory measurements*, NOAA GML, https://gml.noaa.gov/ccgg/trends/ — Mauna Loa monthly means
(Scripps Institution of Oceanography data before May 1974).

In [ ]:
# Setup: upload your own two files from Part A (not someone else's, not newly generated ones)
import os, importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt

NEEDED = ['co2_analysis.py', 'test_co2_analysis.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Not in Colab: put both files next to this notebook.')
    print('Select these files (you can select both at once):', ', '.join(missing))
    files.upload()
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f'Still missing: {missing}. Run this cell again and upload them.'

import co2_analysis as A
importlib.reload(A)
print('Module loaded:', [f for f in dir(A) if not f.startswith('_')])

## Your tests
This cell runs the tests from Part A. All of them must pass before you continue.

In [ ]:
try:
    import pytest
except ImportError:                          # pytest is normally preinstalled in Colab
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'], check=True)
    import pytest
result = pytest.main(['-q', '-p', 'no:cacheprovider', 'test_co2_analysis.py'])
print('All tests passed.' if result == 0 else 'Some tests failed: fix them in Antigravity, upload both files again and rerun.')

## Data

In [ ]:
df_all = A.load_monthly('https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv')
df = df_all[(df_all.year >= 1959) & (df_all.year <= 2025)].reset_index(drop=True)   # complete calendar years
assert len(df) == 67 * 12, 'expected 804 months for 1959-2025'
print(df.head(3)); print('...'); print(df.tail(3))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(df.t, df.co2, lw=.8, label='monthly mean')
ax.plot(df.t, df.co2_des, lw=1.5, label='deseasonalized')
ax.set(xlabel='year', ylabel='CO$_2$ (ppm)', title='Mauna Loa CO$_2$, 1959–2025'); ax.legend(); plt.show()

## Q1 · Straight line or curve? (10 pts)
Fit a degree-1 and a degree-2 polynomial to the **deseasonalized** series with `A.fit_poly` (`t0 = 2000`).

In [ ]:
c1, rms1 = A.fit_poly(df.t, df.co2_des, 1)
c2, rms2 = A.fit_poly(df.t, df.co2_des, 2)
print('linear   ', c1, f'RMS = {rms1:.2f} ppm')
print('quadratic', c2, f'RMS = {rms2:.2f} ppm')

fig, ax = plt.subplots(figsize=(9, 3))
x = df.t - 2000
ax.plot(df.t, df.co2_des - np.polyval(c1, x), lw=.8, label='linear residual')
ax.plot(df.t, df.co2_des - np.polyval(c2, x), lw=.8, label='quadratic residual')
ax.axhline(0, c='k', lw=.5); ax.set(xlabel='year', ylabel='residual (ppm)'); ax.legend(); plt.show()

# TODO: from c2, compute the growth rate implied by the quadratic, d(CO2)/dt, in 1960 and in 2025 (ppm/yr)

**Your answer (Q1):** Give both RMS values with units. Which model describes the data, and what do the residual
plots show? State the quadratic's implied growth rate in 1960 and 2025 and what the coefficient of $(t-2000)^2$ means
physically (with its unit).

## Q2 · Growth rate year by year (10 pts)
`A.jan1_growth` uses NOAA's definition. Compare it with NOAA's own published values.

In [ ]:
g = A.jan1_growth(df_all)
g = g[(g.index >= 1960) & (g.index <= 2025)]
noaa = pd.read_csv('https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_gr_mlo.txt', comment='#', sep=r'\s+', names=['year', 'inc', 'unc']).set_index('year')['inc']
noaa = noaa.reindex(g.index)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(g.index, g, 'o-', ms=3, label='your jan1_growth')
ax.plot(noaa.index, noaa, 's', ms=3, mfc='none', label='NOAA published')
ax.set(xlabel='year', ylabel='growth rate (ppm/yr)'); ax.legend(); plt.show()
print('largest |difference| (ppm/yr):', round((g - noaa).abs().max(), 2))

# TODO: decade means of g for the 1960s, 1970s, ..., 2010s and for 2020-2025
# TODO: fit a straight line to g against year; compare its slope with 2 x (first coefficient of c2)

**Your answer (Q2):** Report the decade means. Is the growth accelerating? Your values and NOAA's are close but
not identical — read NOAA's description at https://gml.noaa.gov/ccgg/trends/gr.html and explain the difference.
Name two years that stand out and propose a physical cause (check it against a source; cite it).

## Q3 · The breathing planet (10 pts)
Remove the quadratic trend from the **monthly means** (`co2`, not `co2_des`) and average by calendar month.

In [ ]:
s60 = A.seasonal_cycle(df, c2, 1960, 1969)
s10 = A.seasonal_cycle(df, c2, 2010, 2019)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(s60.index, s60, 'o-', label='1960s'); ax.plot(s10.index, s10, 's-', label='2010s')
ax.set(xlabel='month', ylabel='detrended CO$_2$ (ppm)', xticks=range(1, 13)); ax.axhline(0, c='k', lw=.5); ax.legend(); plt.show()

# TODO: peak month, trough month and peak-to-trough amplitude for each decade

**Your answer (Q3):** In which months is CO₂ highest and lowest, and why (which hemisphere, which process)?
Did the amplitude change between the 1960s and the 2010s? By how much? Give one possible reason, with a source.

## Q4 · Hidden periods (10 pts)

In [ ]:
resid = df.co2.to_numpy() - np.polyval(c2, df.t.to_numpy() - 2000)
print('strongest periods (months):', [round(p, 1) for p in A.dominant_periods(resid, 3)])

# TODO: plot the periodogram |rfft|^2 against period in months (log x axis), and mark the peaks

**Your answer (Q4):** Explain each of the three periods. One of them is not a natural cycle at all — which one,
and what does it tell you about the quadratic model?

## Q5 · From ppm to gigatonnes of carbon (10 pts)
Derive by hand how many gigatonnes of carbon (GtC) one ppm of CO₂ in the atmosphere represents. Use the mass of the
dry atmosphere, $M = 5.135\times10^{18}$ kg (Trenberth & Smith, *J. Climate* 18, 864, 2005), and the molar mass of
dry air, 28.97 g/mol. Show every step and the units.

In [ ]:
# TODO: compute GtC per ppm from M and the molar masses (do the arithmetic here, not in a chatbot)
GtC_per_ppm = ...

# TODO: carbon added to the atmosphere in 2025, using NOAA's published 2025 growth rate

**Your answer (Q5):** Your factor, compared with the commonly used 2.124 GtC/ppm (what could explain the small
difference?), and the 2025 increase in GtC.

## Verification (10 pts)
Ask the Antigravity agent (or any AI tool) **one numerical question** from this assignment, e.g. *"How many GtC is
1 ppm of atmospheric CO₂?"* or *"What was the CO₂ growth rate at Mauna Loa in 2024?"*. Paste its answer below,
then check it against your own computation **and** a primary source. Was it right? How do you know?

**AI answer (pasted):**

**Your check and source:**

## AI-use disclosure (required)
Three lines, as in Week 4: **Tool** (name and version) · **Purpose** (which part, which task) ·
**Human oversight** (what you checked, what you wrote yourself).

**Tool:**

**Purpose:**

**Human oversight:**